# Fixes test: per-class thresholds, extra inputs and finer STFT bins

Tests the first two fixes from the wider-view test, on all 60 classes with random frequency placement on:

1. **Per-class thresholds.** Each class gets its own threshold, picked on the validation set, alongside the fixed 0.5. Every run below reports both.
2. **Extra inputs and finer bins.** `--features` adds amplitude and instantaneous frequency as two more inputs to the IQ branch. `--stft-nfft 64` makes the STFT bins 50 kHz wide instead of 200 kHz.

The runs use the same quarter of every shard, seed and 15 epochs as the wider-view test, so they compare directly with its `current` model. None of these models is an ensemble member. Don't report their scores, and don't save them to `members/`.

Before you start: **Runtime → Change runtime type → GPU**. The wider-view test's current model took about 15 minutes to train. These runs are slower, so allow 20 to 40 minutes each (a rough guess).

## 1. Get the code

In [ ]:
%cd /content
!rm -rf sedicAI_NEXA
!git clone -b eileen-17class-pilot https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!pip install -q pyyaml h5py gdown

## 2. Check the GPU

This must say `CUDA: True`. If it doesn't, change the runtime type and start again.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Turn on the GPU: Runtime -> Change runtime type'

## 3. Save results to your Drive

`OLD` is the wider-view test's folder: its `current` model is rescored in step 6, not retrained. New runs go to `OUT`, and save a checkpoint every epoch, so a dropped session can carry on.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OLD = '/content/drive/MyDrive/nexa_wide_iq_test'
OUT = '/content/drive/MyDrive/nexa_fixes_test'
!mkdir -p "$OUT/both" "$OUT/features" "$OUT/stft64"
!ls "$OLD/current"

## 4. Download the shards

About 3.8 GB from `processed_data`. Skipped if this session already has them. The training step checks every file against its manifest, so a bad download stops the run.

In [ ]:
import os
FOLDERS = {
    'jam': '1hPb-Hi5-RNXHOPB7eqs5o7oSJ66LZ0Fu',
    'pulse': '18jZ_2gJ1HbCloRw4Au6KbPuUrhJyP3MW',
    'spread': '1xr31dYor-rUnXdyXpK512t-HeGmHyugz',
    'radar': '1rGQmzS3dz0-0GE6Ixk7cVkSUtX6daUel',
    'fskpsk': '15aB-sjz_hWMcejUo0JIwd0S2cackg_Y8',
    'analog': '1oz2amRn65zqJ_JWymViaPhZ0wANdFLuZ',
    'multicarrier': '16FsXoJ4yc--pvm3yv89vd8RhTPh3VR2o',
    'apsk': '1uidz9R9NUz91mYe9ko1o8gBzX5Z8sedr',
}
for group, folder in FOLDERS.items():
    if os.path.isdir(f'/content/shards/{group}') and os.listdir(f'/content/shards/{group}'):
        print(f'{group}: already here')
        continue
    !gdown --folder "https://drive.google.com/drive/folders/{folder}" -O "/content/shards/{group}"
!du -sh /content/shards/*

## 5. Check first

A few minutes on a small sample. If this fails, stop and fix it.

In [ ]:
GROUPS = 'jam pulse spread radar fskpsk analog multicarrier apsk'
RUN = f'python scripts/pilot_train_shards.py --shards /content/shards --groups {GROUPS} --fraction 0.25 --epochs 15'
!python scripts/pilot_train_shards.py --shards /content/shards --groups {GROUPS} --out /content/check --fraction 0.005 --epochs 1 --features --stft-nfft 64

## 6. Rescore the current model (fix 1 alone)

No training: this loads the `current` model from the wider-view test and scores it again with per-class thresholds. It takes a few minutes and rewrites that folder's `scorecard.json`. The numbers at 0.5 stay the same.

In [ ]:
!{RUN} --out "$OLD/current"

## 7. Train

Both changes together first, because that's the main question. If the session drops, run cells 1 to 4 and the `GROUPS` cell again, then the run that stopped: it resumes from your Drive.

In [ ]:
!{RUN} --out "$OUT/both" --features --stft-nfft 64

Then each change alone, to see which one helped and which one hurt. You can skip these if you're short of time, but then a radar or FHSS drop can't be traced to one change.

In [ ]:
!{RUN} --out "$OUT/features" --features

In [ ]:
!{RUN} --out "$OUT/stft64" --stft-nfft 64

## 8. Compare

Recall at the calibrated thresholds only counts if false alarms stay low, so they're shown side by side. The watch list is the classes most likely to suffer from finer bins (FHSS hops) or that the wider view hurt (radar). Tell Claude when this is done and it can read the scorecards from your Drive.

In [ ]:
import json
runs = {'current': f'{OLD}/current', 'both': f'{OUT}/both', 'features': f'{OUT}/features', 'stft64': f'{OUT}/stft64'}
cards = {k: json.load(open(f'{v}/scorecard.json')) for k, v in runs.items() if os.path.exists(f'{v}/scorecard.json')}
print(f"{'run':<10}{'top-1':>8}{'recall 0.5':>12}{'calibrated':>12}{'false alarm':>13}")
for k, c in cards.items():
    print(f"{k:<10}{c['macro_top1']:>8.3f}{c['macro_recall']:>12.3f}"
          f"{c.get('calibrated_macro_recall', float('nan')):>12.3f}{c.get('calibrated_macro_false_alarm', float('nan')):>13.4f}")

WATCH = ['FHSS', 'LFM_RADAR', 'RADAR_FRANK', 'RADAR_BARKER', 'RADAR_PULSE', 'JAM_SPOT', 'JAM_TONE',
         'WBFM', 'NBFM', 'PM', 'RADAR_CW', 'AFSK', '16QAM', '64QAM', '16APSK']
names = list(cards['current']['per_class'])
print()
print('Calibrated recall per class, biggest loss for "both" first')
print(f"{'class':<18}" + ''.join(f'{k:>10}' for k in cards))
cal = lambda k, n: cards[k]['per_class'][n]['calibrated']['recall']
order = sorted(names, key=lambda n: cal('both', n) - cal('current', n)) if 'both' in cards else names
for n in order:
    mark = ' <- watch' if n in WATCH else ''
    print(f'{n:<18}' + ''.join(f'{cal(k, n):>10.3f}' for k in cards) + mark)